# 05 - Predecir una imagen

Objetivo: cargar una imagen individual y revisar clase, confianza, margen y estado.

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MODEL_PATH = PROJECT_ROOT / 'models' / 'modelo_semillas_best.keras'
CLASSES = ['arbejas', 'arroz', 'frijol', 'maiz_pira']
IMG_SIZE = (224, 224)
MIN_CONFIDENCE = 0.70
MIN_MARGIN = 0.20

model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
def predict_image(image_path: Path):
    image = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    array = tf.keras.utils.img_to_array(image)
    batch = np.expand_dims(array, axis=0)
    probabilities = model.predict(batch, verbose=0)[0]
    order = np.argsort(probabilities)[::-1]
    top_idx = int(order[0])
    second_idx = int(order[1])
    confidence = float(probabilities[top_idx])
    margin = float(probabilities[top_idx] - probabilities[second_idx])
    status = 'confiable' if confidence >= MIN_CONFIDENCE and margin >= MIN_MARGIN else 'dudosa'
    return {
        'prediction': CLASSES[top_idx],
        'confidence': confidence,
        'margin': margin,
        'status': status,
        'probabilities': dict(zip(CLASSES, map(float, probabilities)))
    }

# predict_image(PROJECT_ROOT / 'ruta' / 'imagen.jpg')